# Example 1 — Iris Flower Classification

Train a classifier to predict species from flower measurements. No GPU is required.

In [ ]:
%pip -q install pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
print("Setup complete!")

## 1. Load the dataset
We load the public Iris CSV. Each row represents a flower.

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
display(df.head())

## 2. Explore the data
Check data types, missing values, and target balance before training.

In [ ]:
print(df.info())
print("\\nMissing values:")
print(df.isna().sum())
print("\\nSpecies counts:")
print(df["species"].value_counts())

In [ ]:
sns.pairplot(df, hue="species", corner=True)
plt.suptitle("Iris feature relationships", y=1.02)
plt.show()

## 3. Define features and target
`X` contains flower measurements. `y` contains the species we want to predict.

In [ ]:
df = df.dropna(subset=["species"]).copy()
X = df.drop(columns=["species"])
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## 4. Train Logistic Regression
The pipeline imputes missing numeric values, scales measurements, and trains the classifier. Preprocessing is fitted on training data only.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
model.fit(X_train, y_train)
print("Training complete!")

## 5. Evaluate the model

In [ ]:
pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title("Iris confusion matrix")
plt.tight_layout()
plt.show()

## 6. Predict a new flower

In [ ]:
new_flower = pd.DataFrame([{
    "sepal_length": 5.1, "sepal_width": 3.5,
    "petal_length": 1.4, "petal_width": 0.2
}])
print("Predicted species:", model.predict(new_flower)[0])

## 7. Save the model

In [ ]:
joblib.dump(model, "iris_classifier.joblib")
print("Saved iris_classifier.joblib")

## Try it yourself
1. Change the measurements in `new_flower`.
2. Train a `RandomForestClassifier` and compare results.
3. Explain what the confusion matrix tells you.
4. Why must test data remain separate from training?